The spelled-out intro to neural networks and backpropagation - Part 4 [link](https://youtu.be/VMj-3S1tku0?t=6235)

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# @title visulizing the expression
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [3]:
# @title Value class
class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward

    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out

  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  # here we are using tanh as the activation function
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')

    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward

    return out

  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')

    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward

    return out


  def backward(self):

    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

Part 4
---

In [4]:
import torch
import random

In [5]:
# custom neural network classes

class Neuron: # a neuron that can have multiple inputs, weights and a bias

  # param 'nin' is the no of input to a neuron
  def __init__(self, nin):
    # randomely creating weights value and a bias value
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))

  # __call__ allows nueron object to be called like a function
  def __call__(self, x):
    # summazation w * x + b
    act = sum((wi*xi for wi, xi in zip(self.w, x)))+ self.b
    # activation function
    out = act.tanh()
    return out

  # provide parameters(list containing weights and bias) in a neuron
  def parameters(self):
    return self.w + [self.b] # return a list of weights and bias

class Layer: # layer that contains neurons

  # param 'nin' is the no of input to a neuron and 'nout' the no of neurons in this layer
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self, x):
    outs = [n(x) for n in self.neurons] # individuly evulating each neuron
    return outs[0] if len(outs) == 1 else outs

  # iterate over neurons in layer and then iterate over params in each neurons
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP: # Multi layer perceptron , that has layers

  # param 'nin' is the no of input to a neuron and 'nouts' is a list of no of neurons in layer
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

  def __call__(self, x):
    for layer in self.layers: # calling layers sequentially
      x = layer(x)
    return x

  # iterate over layers in MLP and then iterate over params in each layer
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]


In [6]:
# so if we need a MLP with 3 neuron input layer, 2 hidden layer with 4 neurons each and single neuron output layer
x = [2.0, 3.0, -1.0] # inputs
n = MLP(3,[4,4,1])
n(x)


Value(data=-0.8724474018022839)

In [7]:
#draw_dot(n(x))
# n.parameters() # will list all the parameters available for this neural network
len(n.parameters()) # total no of parameters

41

In [8]:
# four inputs
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]

# desired targets/ground truth
ys = [1.0, -1.0, -1.0, 1.0]

In [9]:
ypred = [n(x) for x in xs]
ypred

[Value(data=-0.8724474018022839),
 Value(data=-0.9893109780515008),
 Value(data=-0.870164019147149),
 Value(data=-0.908940287958822)]

----
Find loss

In [10]:
# individual losses . what we need is low losses
[(yout - ygt)**2 for ygt, yout in zip(ys, ypred)]

[Value(data=3.5060592725161235),
 Value(data=0.0001142551902154974),
 Value(data=0.016857381924021884),
 Value(data=3.6440530229923103)]

In [11]:
# loss . here we use mean squared error loss
loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
loss

Value(data=7.167083932622671)

In [12]:
# note: The loss function is the starting point for backpropagation for neural network
# loss.backward()

In [13]:
# an individual weight value
n.layers[0].neurons[0].w[0].data
# an individual weight's grad
n.layers[0].neurons[0].w[0].grad


0.0

----
Minimizing loss/training the neural network

By reducing the loss,we get closer to ground truth

In [14]:
# predition before training
ypred

[Value(data=-0.8724474018022839),
 Value(data=-0.9893109780515008),
 Value(data=-0.870164019147149),
 Value(data=-0.908940287958822)]

In [15]:
# here we are
for k in range(20):

  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

  # backward pass
  for p in n.parameters():
    p.grad = 0.0 # we need do zero grad to rest the grads
  loss.backward()

  # update
  for p in n.parameters():
    # nuding the weight with respect to learning rate(here its 0.1) and grad
    p.data += -0.1 * p.grad # You subtract the grad — bcoz you want to move downhill to reduce the loss.

  print(k, loss.data)

0 7.167083932622671
1 3.8799135123839714
2 3.6055024424120092
3 3.1895485354760367
4 1.9259040493730182
5 1.6893115029155432
6 2.314472563892682
7 0.36278432047416975
8 0.06942313395471128
9 0.05845175868992787
10 0.05040861165506949
11 0.04426787396255308
12 0.03943152010376702
13 0.03552731846379492
14 0.03231168285270079
15 0.02961867775499956
16 0.027331458891797784
17 0.02536542882544347
18 0.023657867207136273
19 0.022161311461544674


In [16]:
# predition after training
ypred

[Value(data=0.9438905232283396),
 Value(data=-0.9610591754335605),
 Value(data=-0.9098952484530961),
 Value(data=0.9031610408528072)]

In [17]:
# this should be final parameter after training
n.parameters()

[Value(data=0.3348885041225227),
 Value(data=0.5785446329377897),
 Value(data=-0.1225267518533821),
 Value(data=0.1883526072158988),
 Value(data=0.809869898657143),
 Value(data=-0.028067913870575417),
 Value(data=-0.546071520581404),
 Value(data=-0.5589040050165592),
 Value(data=0.5972146917359838),
 Value(data=-0.7073613597205956),
 Value(data=0.31477821865387995),
 Value(data=-0.9841683198771847),
 Value(data=-0.059571533123504876),
 Value(data=-0.4531854637674039),
 Value(data=0.8075049423915259),
 Value(data=0.5451039475180808),
 Value(data=-0.36035837603069437),
 Value(data=1.0482660900083607),
 Value(data=-0.622868409865018),
 Value(data=-0.22704619235532034),
 Value(data=-0.1282500977092348),
 Value(data=0.9846666170262057),
 Value(data=0.10910728979082956),
 Value(data=-0.11426419346329739),
 Value(data=-1.041352993863318),
 Value(data=-0.5758742856006053),
 Value(data=0.48707059274169556),
 Value(data=-0.5148674688735234),
 Value(data=0.16636230390800602),
 Value(data=-0.28584